# Patch Generation Notebook: All-Package Diff Generator

**Author:** S Dey | **Last updated:** May 2026

---

## Purpose

Generates git diffs (`--name-only`, `--stat`, `--numstat`, and full `.patch` files)
for all repositories involved in a NuGraph2 port.

Run this **after running the two diagnostic notebooks** and before starting manual porting work.
The `.patch` files produced here are the raw material you apply (with `git apply --reject`)
to each repository in the target version.

## Generalisation

Written for the specific case of diffing the NuGraph2 feature branches
(on `rtriozzi` and `llena` forks) against the v01p01 base tags.
To adapt for a new port:
1. Update `MRB_SOURCE_ROOT` to your source area for the **old** version (the version you are porting FROM)
2. Update `WORKDIR` to where you want the output patch files saved
3. Update each entry in `PACKAGES` — specifically `base_tag` and `feature_ref`
   to reflect the old version tag and the NuGraph feature branch you are diffing from

## Workflow

```
1. Run diagnose_caf_branches.ipynb and diagnose_stage1_labels.ipynb
2. Run this notebook to produce .patch files for all repos
3. For each repo in the target version:
       git apply --reject --whitespace=fix /path/to/full_diff_<repo>.patch
       find . -name '*.rej' | sort   # inspect and resolve any rejects manually
4. Follow the port guide (Part B) for repo-specific manual edits
```

## Output Files

All outputs go to `WORKDIR`:
- `fileListDiffs/<repo>_name_only.txt` — list of changed files
- `fullPatchDiffs/full_diff_<repo>.patch` — full patch for `git apply`
- `diff_summary_all_packages.csv` — summary table (repo, base tag, n files, patch size)


In [10]:
import subprocess
from pathlib import Path

# ── EDIT THESE TWO PATHS ─────────────────────────────────────────────────────
# MRB_SOURCE_ROOT: srcs/ directory of your OLD version dev area (porting FROM)
MRB_SOURCE_ROOT = Path("/exp/icarus/app/users/sdey2/dev/dev_v10_06_00_01p01/srcs")
# WORKDIR: where to write output patch files and summary CSVs
WORKDIR         = Path("/exp/icarus/app/users/sdey2/dev/dev_v10_06_00_04p04_nugraph/portDebugLogsFilesArchive/for_developers/notebooks/patch_generation/outputs")
# ─────────────────────────────────────────────────────────────────────────────

FILELIST_DIR = WORKDIR / "fileListDiffs"
PATCH_DIR    = WORKDIR / "fullPatchDiffs"
FILELIST_DIR.mkdir(parents=True, exist_ok=True)
PATCH_DIR.mkdir(parents=True, exist_ok=True)

# ── PACKAGE DEFINITIONS ──────────────────────────────────────────────────────
# Each entry defines one repository to diff.
# Fields:
#   repo_dir    : subdirectory name under MRB_SOURCE_ROOT
#   base_tag    : clean base tag to diff against (no NuGraph, old version)
#   feature_ref : branch/ref containing NuGraph changes to extract
#   remote_name : (optional) name for a temporary git remote
#   remote_url  : (optional) URL of a fork to add before diffing
#
# For a new port: update base_tag and feature_ref for each package.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess
from pathlib import Path

# ── Edit these two paths ────────────────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────


# ── Package definitions ─────────────────────────────────────────────────────
# Each entry:
#   base_tag      : the tag to diff against (the clean v01p01 base)
#   feature_ref   : the branch/ref that contains NuGraph changes
#   remote_name   : if not None, a remote to add before diffing
#   remote_url    : URL for that remote
#   fetch_before  : if True, run git fetch <remote_name> before diffing

PACKAGES = [
    {
        "repo_dir"    : "icaruscode",
        "base_tag"    : "v10_06_00_01p01",
        "feature_ref" : "feature/rtriozzi_cerati_NuGraph2_Filter",
        "remote_name" : None,
        "remote_url"  : None,
        "fetch_before": False,
    },
    {
        "repo_dir"    : "sbnanaobj",
        "base_tag"    : "v10_00_04",
        "feature_ref" : "feature/rtriozzi_NuGraph2_MoveVars",
        "remote_name" : None,
        "remote_url"  : None,
        "fetch_before": False,
    },
    {
        "repo_dir"    : "sbncode",
        "base_tag"    : "v10_06_00_01",
        "feature_ref" : "feature/llena_CAFMaker_PandoraAfterNuGraph",
        "remote_name" : None,
        "remote_url"  : None,
        "fetch_before": False,
    },
    {
        "repo_dir"    : "lardataobj",
        "base_tag"    : "v10_00_04",
        "feature_ref" : "llena/feature-nugraph-multislice",
        "remote_name" : "llena",
        "remote_url"  : "https://github.com/leonardo-lena/lardataobj.git",
        "fetch_before": True,
    },
    {
        "repo_dir"    : "larrecodnn",
        "base_tag"    : "v10_01_10",
        "feature_ref" : "llena/feature/nugraph-multislice",
        "remote_name" : "llena",
        "remote_url"  : "https://github.com/leonardo-lena/larrecodnn.git",
        "fetch_before": True,
    },
    {
        "repo_dir"    : "larpandora",
        "base_tag"    : "v10_00_19",
        "feature_ref" : "rtriozzi/feature/rtriozzi_NuGraphInterface",
        "remote_name" : "rtriozzi",
        "remote_url"  : "https://github.com/rtriozzi/larpandora.git",
        "fetch_before": True,
    },
    {
        "repo_dir"    : "larpandoracontent",
        "base_tag"    : "v04_15_01",
        "feature_ref" : "rtriozzi/feature/rtriozzi_NuGraphInterface",
        "remote_name" : "rtriozzi",
        "remote_url"  : "https://github.com/rtriozzi/larpandoracontent.git",
        "fetch_before": True,
    },
]

print(f"Packages        : {[p['repo_dir'] for p in PACKAGES]}")

Packages        : ['icaruscode', 'sbnanaobj', 'sbncode', 'lardataobj', 'larrecodnn', 'larpandora', 'larpandoracontent']


In [11]:
import os

def resolve_base_tag(pkg):
    """Return the base tag string, resolving from env var if needed."""
    if pkg.get("base_tag"):
        return pkg["base_tag"]
    env_var = pkg.get("base_tag_env")
    if env_var:
        val = os.environ.get(env_var)
        if val:
            # strip leading 'v' if the env var doesn't include it
            tag = val if val.startswith("v") else f"v{val}"
            return tag
        else:
            raise EnvironmentError(
                f"Environment variable {env_var} is not set. "
                f"Did you source setup_icarus.sh first?"
            )
    raise ValueError(f"No base_tag or base_tag_env defined for {pkg['repo_dir']}")


def run_git(cmd, cwd):
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.returncode != 0:
        print(f"  [ERROR] {' '.join(cmd)}")
        print(f"  stderr: {result.stderr.strip()}")
    return result.stdout, result.returncode


def ensure_remote(repo_path, remote_name, remote_url):
    """Add remote if it doesn't already exist."""
    out, _ = run_git(["git", "remote"], repo_path)
    if remote_name not in out.split():
        print(f"  Adding remote '{remote_name}' → {remote_url}")
        run_git(["git", "remote", "add", remote_name, remote_url], repo_path)
    else:
        print(f"  Remote '{remote_name}' already exists, skipping add.")


def generate_diff(pkg):
    repo_dir  = pkg["repo_dir"]
    repo_path = MRB_SOURCE_ROOT / repo_dir

    if not repo_path.exists():
        print(f"\n[SKIP] {repo_dir}: directory not found at {repo_path}")
        return None

    print(f"\n{'='*60}")
    print(f"Package: {repo_dir}")

    base_tag    = resolve_base_tag(pkg)
    feature_ref = pkg["feature_ref"]
    print(f"  Base tag    : {base_tag}")
    print(f"  Feature ref : {feature_ref}")

    # Ensure remote exists and fetch if needed
    if pkg["remote_name"]:
        ensure_remote(repo_path, pkg["remote_name"], pkg["remote_url"])
        if pkg["fetch_before"]:
            print(f"  Fetching remote '{pkg['remote_name']}'...")
            run_git(["git", "fetch", pkg["remote_name"]], repo_path)

    diff_range = f"{base_tag}..{feature_ref}"

    # -- name-only
    files_changed, rc = run_git(
        ["git", "diff", "--name-only", diff_range], repo_path
    )
    filelist_out = FILELIST_DIR / f"diff_files_{repo_dir}.txt"
    filelist_out.write_text(files_changed)
    n_files = len([l for l in files_changed.splitlines() if l.strip()])
    print(f"  Files changed: {n_files}  → {filelist_out}")

    # -- stat
    stat, _ = run_git(
        ["git", "diff", "--stat", diff_range], repo_path
    )
    print(f"  Stat summary:\n{stat}")

    # -- numstat
    numstat, _ = run_git(
        ["git", "diff", "--numstat", diff_range], repo_path
    )
    numstat_out = FILELIST_DIR / f"numstat_{repo_dir}.txt"
    numstat_out.write_text(numstat)
    print(f"  Numstat saved → {numstat_out}")

    # -- full patch
    full_diff, _ = run_git(
        ["git", "diff", diff_range], repo_path
    )
    patch_out = PATCH_DIR / f"full_diff_{repo_dir}.patch"
    patch_out.write_text(full_diff)
    print(f"  Full patch saved ({len(full_diff):,} chars) → {patch_out}")

    return {
        "repo"        : repo_dir,
        "base_tag"    : base_tag,
        "feature_ref" : feature_ref,
        "n_files"     : n_files,
        "patch_size_b": len(full_diff.encode()),
        "patch_path"  : str(patch_out),
        "filelist_path": str(filelist_out),
    }

print("Helper functions defined.")

Helper functions defined.


## Run Diff Generation

Runs the diff for all packages defined above.
This may take a moment for large repos like icaruscode.
Output patch files are written to `WORKDIR/fullPatchDiffs/`.


In [12]:
# ── Run for all packages ────────────────────────────────────────────────────
results = []
for pkg in PACKAGES:
    result = generate_diff(pkg)
    if result:
        results.append(result)

print(f"\n{'='*60}")
print(f"Done. Processed {len(results)}/{len(PACKAGES)} packages.")


Package: icaruscode
  Base tag    : v10_06_00_01p01
  Feature ref : feature/rtriozzi_cerati_NuGraph2_Filter
  Files changed: 38  → /exp/icarus/app/users/sdey2/dev/dev_v10_06_00_04p04_nugraph/portDebugLogsFilesArchive/for_developers/notebooks/patch_generation/outputs/fileListDiffs/diff_files_icaruscode.txt
  Stat summary:
 CMakeLists.txt                                     |   1 +
 fcl/caf/cafmaker_defs.fcl                          |   3 +
 fcl/caf/cafmakerjob_icarus_data_NuGraphReco.fcl    |  43 ++
 .../cafmakerjob_icarus_detsim2d_NuGraphReco.fcl    |  13 +
 ..._detsim2d_systtools_and_fluxwgt_NuGraphReco.fcl |  13 +
 fcl/reco/Definitions/stage1_icarus_defs.fcl        | 153 +++--
 fcl/reco/Stage1/data/stage1_run2_1d_icarus.fcl     |   1 +
 .../data/stage1_run2_1d_icarus_NuGraphReco.fcl     |  13 +
 fcl/reco/Stage1/data/stage1_run2_icarus.fcl        |   1 +
 .../Stage1/data/stage1_run2_icarus_NuGraphReco.fcl |  81 +++
 fcl/reco/Stage1/mc/stage1_run2_1d_icarus_MC.fcl    |   3 +-
 .../mc/

## Summary Table

Displays a summary of all processed packages: files changed and patch size.
Use this to gauge the scope of the port before applying any patches.

**Reference output for v01p01 → v04p04:**

| Repo | Files changed | Patch size | Risk |
|---|---|---|---|
| icaruscode | 38 | 163.6 KB | Medium-High |
| sbncode | 4 | 22.7 KB | Medium-High |
| larpandora | 6 | 11.0 KB | Medium |
| larpandoracontent | 1 | 9.6 KB | Low |
| sbnanaobj | 5 | 5.6 KB | Low-Medium |
| lardataobj | 0 | — | None (already upstream) |
| larrecodnn | 0 | — | None (already upstream) |


In [13]:
# ── Summary table ───────────────────────────────────────────────────────────
import pandas as pd

summary_df = pd.DataFrame(results)[[
    "repo", "base_tag", "feature_ref", "n_files", "patch_size_b"
]].copy()
summary_df["patch_size_kb"] = (summary_df["patch_size_b"] / 1024).round(1)
summary_df = summary_df.drop(columns=["patch_size_b"])

summary_df.to_csv(WORKDIR / "diff_summary_all_packages.csv", index=False)
display(summary_df)
print(f"\nSummary saved → {WORKDIR / 'diff_summary_all_packages.csv'}")

,repo,base_tag,feature_ref,n_files,patch_size_kb
0,icaruscode,v10_06_00_01p01,feature/rtriozzi_cerati_NuGraph2_Filter,38,163.6
1,sbnanaobj,v10_00_04,feature/rtriozzi_NuGraph2_MoveVars,5,5.6
2,sbncode,v10_06_00_01,feature/llena_CAFMaker_PandoraAfterNuGraph,4,22.7
3,lardataobj,v10_00_04,llena/feature-nugraph-multislice,0,0.0
4,larrecodnn,v10_01_10,llena/feature/nugraph-multislice,0,0.0
5,larpandora,v10_00_19,rtriozzi/feature/rtriozzi_NuGraphInterface,6,11.0
6,larpandoracontent,v04_15_01,rtriozzi/feature/rtriozzi_NuGraphInterface,1,9.6



Summary saved → /exp/icarus/app/users/sdey2/dev/dev_v10_06_00_04p04_nugraph/portDebugLogsFilesArchive/for_developers/notebooks/patch_generation/outputs/diff_summary_all_packages.csv


## Per-Package File List Preview

Sanity check before applying patches.
If a file you did not expect appears here, investigate before proceeding.

**After this notebook**, apply each patch to the corresponding repo in your target version:

```bash
cd $MRB_SOURCE/<repo>
git apply --reject --whitespace=fix /path/to/fullPatchDiffs/full_diff_<repo>.patch
find . -name '*.rej' | sort   # list rejects to resolve manually
```

Expect rejects for heavily modified files (e.g. `stage1_icarus_defs.fcl`,
`CAFMaker_module.cc`). These require manual porting — see Part B of the port guide.


In [14]:
# ── Per-package file list preview ───────────────────────────────────────────
# Useful sanity check before uploading patches
for r in results:
    print(f"\n{'─'*50}")
    print(f"{r['repo']} ({r['n_files']} files changed):")
    txt = Path(r['filelist_path']).read_text()
    for line in txt.splitlines():
        print(f"  {line}")


──────────────────────────────────────────────────
icaruscode (38 files changed):
  CMakeLists.txt
  fcl/caf/cafmaker_defs.fcl
  fcl/caf/cafmakerjob_icarus_data_NuGraphReco.fcl
  fcl/caf/cafmakerjob_icarus_detsim2d_NuGraphReco.fcl
  fcl/caf/cafmakerjob_icarus_detsim2d_systtools_and_fluxwgt_NuGraphReco.fcl
  fcl/reco/Definitions/stage1_icarus_defs.fcl
  fcl/reco/Stage1/data/stage1_run2_1d_icarus.fcl
  fcl/reco/Stage1/data/stage1_run2_1d_icarus_NuGraphReco.fcl
  fcl/reco/Stage1/data/stage1_run2_icarus.fcl
  fcl/reco/Stage1/data/stage1_run2_icarus_NuGraphReco.fcl
  fcl/reco/Stage1/mc/stage1_run2_1d_icarus_MC.fcl
  fcl/reco/Stage1/mc/stage1_run2_1d_icarus_NuGraphReco_MC.fcl
  fcl/reco/Stage1/mc/stage1_run2_icarus_MC.fcl
  fcl/reco/Stage1/mc/stage1_run2_icarus_NuGraphReco_MC.fcl
  fcl/reco/Stage1/overlay/stage1_run2_icarus_overlay_NuGraphReco.fcl
  icaruscode/TPC/CMakeLists.txt
  icaruscode/TPC/NuGraph/CMakeLists.txt
  icaruscode/TPC/NuGraph/ICARUSFilteredNuSliceHitsProducer_module.cc
  ic